In [1]:
import warnings

warnings.filterwarnings("ignore")

## State

- 그래프 전체에서 공유하는 공용 데이터 저장소
- Node는 State를 읽고 새로운 값을 반환하며 반환된 값이 다음 Node의 State에 반영된다


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    name: str
    age: int


def introduce(state: State):
    print("나이: ", state["age"])
    print("이름: ", state["name"])
    return {"age": state["age"] + 1}


builder = StateGraph(State)

builder.add_node("introduce", introduce)

builder.add_edge(START, "introduce")
builder.add_edge("introduce", END)

graph = builder.compile()

result = graph.invoke({"name": "신형만", "age": 32})

print(result)

나이:  32
이름:  신형만
{'name': '신형만', 'age': 33}


## Node

- Node는 실제로 작업을 수행하는 함수
- LLM 호출, DB 조회, 검색, 계산 API 호출 등 실제 대부분의 작업은 Node 안에서 수행


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    number: int
    result: int


def multiply_node(state: State):
    result = state["number"] * 10

    return {"result": result}


builder = StateGraph(State)

builder.add_node("multiply", multiply_node)

builder.add_edge(START, "multiply")

builder.add_edge("multiply", END)

graph = builder.compile()

result = graph.invoke({"number": 7, "result": 0})

print(result)

{'number': 7, 'result': 70}


## Edge

- Edge는 어떤 Node 다음에 어떤 Node를 실행할 지 결정하는 연결선이다


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    message: str


def node_a(state: State):
    print("Node A 실행")

    return {"message": state["message"] + " -> A"}


def node_b(state: State):
    print("Node B 실행")

    return {"message": state["message"] + " -> B"}


builder = StateGraph(State)

builder.add_node("a", node_a)
builder.add_node("b", node_b)

builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

graph = builder.compile()

result = graph.invoke({"message": "START"})

print(result)

Node A 실행
Node B 실행
{'message': 'START -> A -> B'}


## Conditional Edge

- Conditional Edge는 State의 값을 보고 다음 Node를 "동적"으로 결정


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    score: int
    result: str


def check_score(state: State):
    return {}


def decide_next(state: State):
    if state["score"] >= 80:
        return "pass"
    return "fail"


def pass_node(state: State):
    return {"result": "합격"}


def fail_node(state: State):
    return {"result": "불합격"}


builder = StateGraph(State)

builder.add_node("check", check_score)
builder.add_node("decide", decide_next)
builder.add_node("pass", pass_node)
builder.add_node("fail", fail_node)

builder.add_edge(START, "check")
builder.add_edge("pass", END)
builder.add_edge("fail", END)

builder.add_conditional_edges("check", decide_next, {"pass": "pass", "fail": "fail"})

graph = builder.compile()

result = graph.invoke({"score": 80, "result": ""})
# result = graph.invoke({"score": 75, "result": ""})

print(result)

{'score': 80, 'result': '합격'}


## Loop

- 조건부 엣지(Conditional Edge)를 사용하여 특정 노드가 이전 노드로 다시 돌아가도록 순환 구조를 구성하여 구현


In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    count: int
    result: str
    message: str


def work(state: State):
    count = state["count"] + 1
    return {"count": count, "message": state["message"] + " -> work"}


def decide(state: State):
    new_message = state["message"] + " -> decide"
    if state["count"] >= 3:
        return {"result": "finish", "message": new_message}
    return {"result": "continue", "message": new_message}


def route(state: State) -> str:
    return state["result"]


builder = StateGraph(State)

builder.add_node("work", work)
builder.add_node("decide", decide)


builder.add_edge(START, "work")
builder.add_edge("work", "decide")


builder.add_conditional_edges("decide", route, {"continue": "work", "finish": END})

graph = builder.compile()

result = graph.invoke({"count": 0, "result": "", "message": "START"})

print(result["message"])

START -> work -> decide -> work -> decide -> work -> decide


## Reducer And MessagesState

- add_messages라는 reducer를 이용해 새로운 메세지를 기존 메세지에 자연스럽게 합칠 수 있다


In [7]:
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from typing import Annotated, TypedDict
# from langgraph.graph.message import add_messages


# Annotated에 Reducer를 활용하여 사용 가능 MessagesState는 내부적으로 Reducer가 포함되어 있음
# class State(TypedDict):
#     messages: Annotated[list[BaseMessage], add_messages]


def chatbot(state: MessagesState):
    last_message = state["messages"][-1]

    response = AIMessage(content=f"받은 메세지 : {last_message}")

    return {"messages": response}


builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage(content="LangGraph가 무엇인가요")]})
# result = graph.invoke(
#     {"messages": [{"role": "user", "content": "LangChain가 무엇인가요"}]}
# )

for message in result["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : LangGraph가 무엇인가요
AIMessage : 받은 메세지 : content='LangGraph가 무엇인가요' additional_kwargs={} response_metadata={} id='5072fd07-293f-4e40-89fa-462cf60bd8c1'


## Command

- State 업데이트와 다음 실행 위치 이동을 하나의 Node에서 동시에 제어할 때 사용
- Conditional Edge가 그래프 밖에서 분기 결정을 한다면 Command는 Node 내부에서 State 변경 및 이동을 같이 제어


In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class State(TypedDict):
    score: int
    result: str


def evaluate(state: State):
    if state["score"] >= 80:
        return Command(update={"result": "합격"}, goto="pass")
    return Command(update={"result": "불합격"}, goto="fail")


def pass_node(state: State):
    print("합격 처리")
    return {}


def fail_node(state: State):
    print("불합격 처리")
    return {}


builder = StateGraph(State)

builder.add_node("evaluate", evaluate)
builder.add_node("pass", pass_node)
builder.add_node("fail", fail_node)

builder.add_edge(START, "evaluate")
builder.add_edge("pass", END)
builder.add_edge("fail", END)

graph = builder.compile()

result = graph.invoke({"score": 80, "result": ""})
# result = graph.invoke({"score": 70, "result": ""})

print(result)

합격 처리
{'score': 80, 'result': '합격'}


## LangGraph + LLM


In [9]:
from const.const import create_model

from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# 본인 상황에 맞는 LLM으로 Logic 수정
llm = create_model()


def chatbot(state: MessagesState):
    print("LLM 호출 시작")
    response = llm.invoke(state["messages"])
    print("LLM 호출 완료")

    return {"messages": [response]}


builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

result = graph.invoke(
    {"messages": [HumanMessage(content="LangGraph를 한 문장으로 설명해주세요.")]}
)

print(result["messages"][-1].content)

LLM 호출 시작
LLM 호출 완료
LangGraph는 상태(state)를 유지하는 그래프 구조(노드와 엣지)를 기반으로 LLM 에이전트가 반복, 조건 분기, 다중 에이전트 협업 등 복잡한 워크플로우를 제어 가능하게 구축할 수 있도록 해주는 라이브러리입니다.


## Tool Calling Agent

- LLM이 외부 도구(API, DB 등)를 직접 선택하고 호출하도록 구조화된 그래프를 만드는 것
- Agent는 LLM이 상황에 따라 직접 답변하거나 Tool을 사용할 지 결정하는 구조이고 아래는 그것을 LangGraph에서 Node와 Conditional Edge로 표현


In [10]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from const.const import create_model

llm = create_model()


@tool
def calculator(expression: str) -> str:
    """간단한 수학식을 계산합니다."""

    try:
        result = eval(expression, {"__builtin__": {}}, {})

        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"


tools = [calculator]

llm_with_tools = llm.bind_tools(tools)


def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])

    return {"messages": response}


def tool_node(state: MessagesState):
    last_message = state["messages"][-1]

    tool_messages = []

    for tool_call in last_message.tool_calls:
        if tool_call["name"] == "calculator":
            result = calculator.invoke(tool_call["args"])

            tool_messages.append(
                ToolMessage(content=result, tool_call_id=tool_call["id"])
            )

    return {"messages": tool_messages}


def route_after_agent(state: MessagesState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


builder = StateGraph(MessagesState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})

builder.add_edge("tools", "agent")

graph = builder.compile()

result = graph.invoke(
    {"messages": [HumanMessage(content="123 곱하기 456을 계산해주세요.")]}
)

for message in result["messages"]:
    print(type(message).__name__, ":", message.content)

HumanMessage : 123 곱하기 456을 계산해주세요.
AIMessage : 
ToolMessage : 56088
AIMessage : 123 곱하기 456의 계산 결과는 **56,088**입니다.


## Agent Loop

- Agent란 사실 LLM -> Tool -> LLM -> Tool ... -> 종료의 반복이다
- LangGraph에선 이 실행 Loop를 직접 제어할 수 있다


In [11]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from const.const import create_model

llm = create_model()


@tool
def add_numbers(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b


@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 숫자를 곱합니다."""
    return a * b


tools = {"add_numbers": add_numbers, "multiply_numbers": multiply_numbers}

llm_with_tools = llm.bind_tools(list(tools.values()))


def agent(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])

    return {"messages": response}


def execute_tools(state: MessagesState):
    last_message = state["messages"][-1]

    results = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]

        tool_instance = tools[tool_name]

        result = tool_instance.invoke(tool_call["args"])

        results.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    return {"messages": results}


def route(state: MessagesState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


builder = StateGraph(MessagesState)

builder.add_node("agent", agent)
builder.add_node("tools", execute_tools)

builder.add_edge(START, "agent")

builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
builder.add_edge("tools", END)

graph = builder.compile()

result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="10 더하기 20을 계산하고, 그 결과에 5를 곱해주세요.")
        ]
    }
)


for message in result["messages"]:
    print(f"{type(message).__name__}: {message.content}")

HumanMessage: 10 더하기 20을 계산하고, 그 결과에 5를 곱해주세요.
AIMessage: 먼저 10 더하기 20을 계산하겠습니다.
ToolMessage: 30


## Persistence / Checkpointer

- Checkpointer는 Graph 실행 중 State를 저장하여 대화 기억 중단 후 재개 Thread별 상태 관리를 가능하게 한다


In [12]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState
from const.const import create_model

llm = create_model()


def chatbot(state: MessagesState):
    response = llm.invoke(state["messages"])

    return {"messages": response}


builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

check_pointer = InMemorySaver()

graph = builder.compile(checkpointer=check_pointer)

config = {"configurable": {"thread_id": "user_001"}}

result_1 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름은 코난. 탐정이죠.")]}, config=config
)

result_2 = graph.invoke(
    {"messages": [HumanMessage(content="방금 제가 뭐라고 했나요?")]}, config=config
)

print("첫 번째 실행:")
print(result_1["messages"][-1].content)

print("\n두 번째 실행:")
print(result_2["messages"][-1].content)

첫 번째 실행:
진실은 언제나 하나! 

어머, 명탐정 코난님이시군요. 어디선가 들어본 이름 같은데... 혹시 쿠도 신이치님이신가요?

오늘은 어떤 사건을 맡으러 오셨어요? 아니면 제가 용의자가 된 걸까요? 알리바이 준비해야 할까요? 😄

두 번째 실행:
방금 말씀하신 건: **"내 이름은 코난. 탐정이죠."** 였습니다.

명탐정이 저를 시험해 보신 건가요? 저는 검은 조직이 아니라서 기억력이 꽤 좋습니다. 😄

혹시 이번 사건의 첫 번째 단서였던 걸까요?


## Human-in-the-Loop

- LangGraph에서 실행을 특정 지점에서 사람의 승인이나 입력을 기다리도록 중단하는 구조


In [13]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command


class State(TypedDict):
    message: str
    approved: str
    result: str


def prepare(state: State):
    return {"result": f"승인 요청 내용: {state['message']}"}


def human_approval(state: State):
    answer = interrupt(
        {"question": "이 작업을 승인하시겠습니까?", "message": state["result"]}
    )

    return {"approved": answer}


def process(state: State):
    user_approval = str(state["approved"]).strip().lower()

    if user_approval in ["yes", "y", "true", "승인"]:
        return {"result": "승인되어 작업을 실행했습니다."}

    return {"result": "승인이 거부되었습니다."}


builder = StateGraph(State)

builder.add_node("prepare", prepare)
builder.add_node("approval", human_approval)
builder.add_node("process", process)

builder.add_edge(START, "prepare")
builder.add_edge("prepare", "approval")
builder.add_edge("approval", "process")
builder.add_edge("process", END)

check_pointer = InMemorySaver()

graph = builder.compile(checkpointer=check_pointer)

config = {"configurable": {"thread_id": "approval-001"}}

first_result = graph.invoke(
    {"message": "중요한 작업을 실행합니다.", "approved": "", "result": ""},
    config=config,
)

user_input = input("작업을 승인하시겠습니까? (yes/no): ")

second_result = graph.invoke(Command(resume=user_input), config=config)

print("=== 최종 처리 결과 ===")
print(second_result["result"])

=== 최종 처리 결과 ===
승인되어 작업을 실행했습니다.


## Streaming

- Graph가 실행되는 동안 Node별 진행 상황이나 LLM 출력 등을 실시간으로 전달

- updates
  → Node가 업데이트한 State

- values
  → 전체 State

- messages
  → LLM 메시지 스트리밍

- custom
  → 개발자가 직접 보내는 이벤트


In [14]:
from const.const import create_model
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState

llm = create_model()


def generator(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


def reviewer(state: MessagesState):
    last_message = state["messages"][-1]
    prompt = f"다음 답변을 간단히 한 줄로 요약해줘: {last_message.content}"
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"messages": [response]}


builder = StateGraph(MessagesState)

builder.add_node("generator", generator)
builder.add_node("reviewer", reviewer)

builder.add_edge(START, "generator")
builder.add_edge("generator", "reviewer")
builder.add_edge("reviewer", END)

graph = builder.compile()

for chunk in graph.stream(
    {"messages": [HumanMessage(content="인공지능의 역사에 대해 짧게 설명해줘.")]},
    stream_mode="updates",
):
    for node_name, node_update in chunk.items():
        print(f"[{node_name}]")
        print(node_update["messages"][-1].content)
        print("=" * 50)

[generator]
# 인공지능의 역사

**1950년대 – 탄생**
앨런 튜링이 "기계는 생각할 수 있는가?"를 제기했고(1950), 1956년 다트머스 회의에서 '인공지능(AI)'이라는 용어가 처음 사용되며 학문으로 시작되었습니다.

**1960~70년대 – 초기 발전과 첫 번째 침체**
ELIZA 같은 초기 프로그램이 등장했지만 기대만큼 성과를 내지 못하면서 연구비가 축소되는 'AI 겨울'이 찾아왔습니다.

**1980년대 – 전문가 시스템**
규칙 기반의 전문가 시스템이 상용화되며 붐이 일었으나, 한계가 드러나며 다시 침체기가 이어졌습니다.

**1990~2000년대 – 머신러닝의 부상**
1997년 IBM의 딥블루가 체스 챔피언 카스파로프를 꺾었고, 통계 기반 학습 방법이 주류가 되었습니다.

**2010년대 – 딥러닝 혁명**
2012년 이미지 인식 대회에서 딥러닝(AlexNet)이 압도적 성능을 보이며 돌파구가 열렸고, 2016년 알파고가 이세돌 9단을 꺾었습니다. 2017년 트랜스포머 기술이 등장해 현대 AI의 기반이 마련되었습니다.

**2020년대 – 생성형 AI 시대**
ChatGPT 등 대규모 언어모델이 대중화되면서 AI가 일상 속으로 빠르게 확산되고 있습니다.

핵심 흐름은 **규칙 기반 → 학습 기반 → 딥러닝 → 생성형 AI**로 진화해 온 것이라고 요약할 수 있습니다.
[reviewer]
인공지능은 1950년대 탄생 이후 규칙 기반 → 학습 기반 → 딥러닝을 거쳐 오늘날 생성형 AI 시대로 진화해 왔습니다.


## Subgraph

- 큰 Graph 안에 작은 Graph를 하나의 Node처럼 삽입하는 구조
- 복잡한 Agent 시스템을 모듈화할 때 중요


In [15]:
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from const.const import create_model

llm = create_model()


class ChildState(TypedDict):
    topic: str
    draft: str
    refined_draft: str


def generate_draft(state: ChildState):
    prompt = f"'{state['topic']}' 주제에 대해 핵심적인 내용 2문장을 작성해주세요."
    response = llm.invoke([HumanMessage(content=prompt)])

    draft_text = str(response.content).strip()
    print("generate_draft : ", draft_text)
    return {"draft": draft_text}


def refine_draft(state: ChildState):
    messages = [
        SystemMessage(
            content="당신은 글을 다듬는 에디터입니다. 부연 설명이나 JSON 형식 없이, 입력된 글을 바탕으로 교정된 텍스트만 출력하세요."
        ),
        HumanMessage(
            content=f"다음 글을 읽기 쉬운 블로그 포스팅 톤으로 자연스럽게 교정해주세요:\n\n{state['draft']}"
        ),
    ]

    response = llm.invoke(messages)
    refined_text = str(response.content).strip()
    print("refine_draft : ", refined_text)
    return {"refined_draft": refined_text}


child_builder = StateGraph(ChildState)

child_builder.add_node("generate_draft", generate_draft)
child_builder.add_node("refine_draft", refine_draft)

child_builder.add_edge(START, "generate_draft")
child_builder.add_edge("generate_draft", "refine_draft")
child_builder.add_edge("refine_draft", END)

child_graph = child_builder.compile()


class ParentState(TypedDict):
    topic: str
    final_result: str


def run_child(state: ParentState):
    child_result = child_graph.invoke(
        {"topic": state["topic"], "draft": "", "refined_draft": ""}
    )

    return {"final_result": child_result["refined_draft"]}


parent_builder = StateGraph(ParentState)

parent_builder.add_node("child_workflow", run_child)

parent_builder.add_edge(START, "child_workflow")
parent_builder.add_edge("child_workflow", END)

parent_graph = parent_builder.compile()

result = parent_graph.invoke({"topic": "인공지능의 미래", "final_result": ""})

print("\n=== 최종 실행 결과 ===")
print(result)

generate_draft :  인공지능은 생성형 AI와 자율 시스템의 급속한 발전을 통해 단순 자동화를 넘어 의료, 교육, 과학 연구 등 전 산업의 패러다임을 바꾸는 핵심 동력으로 자리잡을 것입니다. 다만 이러한 기술적 진보와 함께 일자리 구조의 재편, 알고리즘 편향, 프라이버시 침해 등 윤리적·사회적 문제를 해결하기 위한 제도적 장치와 인공지능 거버넌스 확립이 미래 사회의 중요한 과제로 대두될 것입니다.
refine_draft :  인공지능은 생성형 AI와 자율 시스템의 빠른 발전 덕분에, 이제 단순한 자동화를 넘어 의료, 교육, 과학 연구 등 모든 산업의 패러다임을 바꾸는 핵심 동력으로 자리잡고 있어요. 그런데 기술이 발전할수록 새로운 고민도 함께 생기죠. 일자리 구조의 변화, 알고리즘 편향, 프라이버시 침해 같은 윤리적·사회적 문제들이 바로 그것인데요. 그래서 이런 문제를 해결하기 위한 제도적 장치를 마련하고 인공지능 거버넌스를 확립하는 일이 앞으로 미래 사회의 중요한 과제가 될 것 같아요.

=== 최종 실행 결과 ===
{'topic': '인공지능의 미래', 'final_result': '인공지능은 생성형 AI와 자율 시스템의 빠른 발전 덕분에, 이제 단순한 자동화를 넘어 의료, 교육, 과학 연구 등 모든 산업의 패러다임을 바꾸는 핵심 동력으로 자리잡고 있어요. 그런데 기술이 발전할수록 새로운 고민도 함께 생기죠. 일자리 구조의 변화, 알고리즘 편향, 프라이버시 침해 같은 윤리적·사회적 문제들이 바로 그것인데요. 그래서 이런 문제를 해결하기 위한 제도적 장치를 마련하고 인공지능 거버넌스를 확립하는 일이 앞으로 미래 사회의 중요한 과제가 될 것 같아요.'}


## Multi-Agent

- 하나의 거대한 Agent 대신 역할별 Agent를 여러 개 만들고 Supervisor가 작업을 분배하는 구조


In [18]:
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

from const.const import create_model


llm = create_model()


class State(TypedDict):
    task: str
    next_agent: str
    result: str


def supervisor(state: State):
    prompt = (
        "당신은 작업 요청을 분석하여 올바른 전문 에이전트에게 "
        "할당하는 감독관(Supervisor)입니다.\n"
        "사용자의 작업 요청을 분석하고 다음 세 가지 분류 중 "
        "하나만 단어로 출력하세요.\n\n"
        "분류 기준:\n"
        "- research: 정보 검색, 조사, 자료 탐색 및 분석 관련 요청\n"
        "- calculator: 수학 계산, 수식, 수치 연산 관련 요청\n"
        "- unknown: 위 두 가지에 해당하지 않는 기타 일반 대화 및 "
        "분류 불가능한 요청\n\n"
        "오직 'research', 'calculator', 'unknown' 중 하나만 답하세요."
    )

    response = llm.invoke(
        [
            SystemMessage(content=prompt),
            HumanMessage(content=state["task"]),
        ]
    )

    agent_choice = str(response.content).strip().lower()

    if "research" in agent_choice:
        selected = "research"
    elif "calculator" in agent_choice:
        selected = "calculator"
    else:
        selected = "unknown"

    print(f"[Supervisor] 선택된 에이전트: {selected}")

    return {"next_agent": selected}


def research_agent(state: State):
    print("[Research Agent] 정보 조사 작업을 수행 중입니다...")

    prompt = "다음 요청에 대해 전문 정보 조사 에이전트로서 답변해주세요."

    response = llm.invoke(
        [
            SystemMessage(content=prompt),
            HumanMessage(content=state["task"]),
        ]
    )

    result_text = str(response.content).strip()

    if not result_text:
        result_text = "조사 결과를 생성하지 못했습니다."

    return {"result": result_text}


def calculator_agent(state: State):
    print("[Calculator Agent] 계산 작업을 수행 중입니다...")

    prompt = (
        "다음 요청에 대해 수학 계산 전문 에이전트로서 "
        "계산 과정과 결과값을 명확히 작성해주세요."
    )

    response = llm.invoke(
        [
            SystemMessage(content=prompt),
            HumanMessage(content=state["task"]),
        ]
    )

    result_text = str(response.content).strip()

    if not result_text:
        result_text = "계산 결과를 생성하지 못했습니다."

    return {"result": result_text}


def unknown_agent(state: State):
    print("[Unknown Agent] 일반 문의 응대를 수행 중입니다...")

    prompt = "다음 일반 문의에 대해 친절하게 답변해주세요."

    response = llm.invoke(
        [
            SystemMessage(content=prompt),
            HumanMessage(content=state["task"]),
        ]
    )

    result_text = str(response.content).strip()

    if not result_text:
        result_text = "답변을 생성하지 못했습니다."

    return {"result": result_text}


def route(state: State):
    return state["next_agent"]


builder = StateGraph(State)

builder.add_node("supervisor", supervisor)
builder.add_node("research", research_agent)
builder.add_node("calculator", calculator_agent)
builder.add_node("unknown", unknown_agent)

builder.add_edge(START, "supervisor")

builder.add_conditional_edges(
    "supervisor",
    route,
    {
        "research": "research",
        "calculator": "calculator",
        "unknown": "unknown",
    },
)

builder.add_edge("research", END)
builder.add_edge("calculator", END)
builder.add_edge("unknown", END)

graph = builder.compile()


result = graph.invoke(
    {
        "task": "123 곱하기 321 더하기 50 나누기 3을 계산해주세요.",
        "next_agent": "",
        "result": "",
    }
)


print("\n=== 최종 처리 결과 ===")
print(result["result"])

[Supervisor] 선택된 에이전트: calculator
[Calculator Agent] 계산 작업을 수행 중입니다...

=== 최종 처리 결과 ===
# 계산 결과

## 계산식
**123 × 321 + 50 ÷ 3**

연산 순서에 따라 곱셈·나눗셈을 먼저 계산합니다.

---

## 계산 과정

**1단계: 곱셈 (123 × 321)**
```
  123 × 300 = 36,900
  123 ×  20 =  2,460
  123 ×   1 =    123
  ──────────────────
  36,900 + 2,460 + 123 = 39,483
```

**2단계: 나눗셈 (50 ÷ 3)**
```
  50 ÷ 3 = 16.6666... (순환소수, 16⅔)
```

**3단계: 덧셈 (39,483 + 16.6666...)**
```
  39,483 + 16⅔ = 39,499⅔
```

분수로 계산하면:
```
  39,483 + 50/3 = (39,483 × 3 + 50) / 3 = 118,499 / 3
```

---

## 최종 결과

| 표현 형태 | 값 |
|---|---|
| **분수** | 118,499 / 3 |
| **대분수** | 39,499 ⅔ |
| **소수 (반올림)** | 약 **39,499.67** |

✅ **답: 39,499⅔ (≈ 39,499.67)**


## RAG + LangGraph


In [21]:
from typing import TypedDict
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from const.const import create_model, create_embedding

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = create_model()

embeddings = create_embedding()

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)


class State(TypedDict):
    question: str
    documents: list[Document]
    answer: str


def retrieve(state: State):
    print(f"[Retrieve] 질문: '{state['question']}' 에 대한 연관 문서 검색 중...")
    docs = hybrid_retriever.invoke(state["question"])
    print(f"[Retrieve] {len(docs)}개의 연관 문서 검색 완료")
    return {"documents": docs}


def generate(state: State):
    print("[Generate] 검색된 문서를 바탕으로 LLM 답변 생성 중...")
    if not state["documents"]:
        return {"answer": "관련 문서를 찾지 못했습니다."}

    context = "\n".join(doc.page_content for doc in state["documents"])

    prompt = (
        "당신은 금융 안전 전문 상담원입니다.\n"
        "아래 제공된 [참고 문서] 내용을 바탕으로 사용자의 질문에 정확하고 친절하게 답변해주세요.\n"
        "참고 문서에 기재된 내용에만 기반하여 답변하고, 근거 없는 내용은 지어내지 마세요.\n\n"
        f"[참고 문서]\n{context}\n\n"
    )

    response = llm.invoke(
        [SystemMessage(content=prompt), HumanMessage(content=state["question"])]
    )
    return {"answer": response.content}


builder = StateGraph(State)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

graph = builder.compile()

result = graph.invoke(
    {
        "question": "모바일 금융거래를 할 때 어떤 점을 주의해야 하나요?",
        "documents": [],
        "answer": "",
    }
)

print("\n=== 최종 RAG 생성 답변 ===")
print(result["answer"])

[Retrieve] 질문: '모바일 금융거래를 할 때 어떤 점을 주의해야 하나요?' 에 대한 연관 문서 검색 중...
[Retrieve] 10개의 연관 문서 검색 완료
[Generate] 검색된 문서를 바탕으로 LLM 답변 생성 중...

=== 최종 RAG 생성 답변 ===
모바일 금융거래 시에는 보안에 각별한 신경을 쓰셔야 합니다. 특히 돈과 관련된 금융거래에서는 모바일폰의 보안이 무엇보다도 중요합니다. 참고 문서에 제시된 **「모바일폰 금융거래 10계명」**을 안내해 드립니다.

## 모바일폰 금융거래 10계명

1. **공식 배포처 확인** – 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스를 이용하세요.
2. **금융정보 저장 금지** – 모바일폰이나 인터넷에 금융정보를 저장하지 마세요.
3. **비밀번호 안전 관리** – 금융거래 비밀번호를 안전하게 관리하세요.
4. **분실·도난 시 대응** – 모바일폰 분실·도난 시 즉시 모바일폰 금융서비스 사용을 중지하세요.
5. **교체·수리 전 정보 삭제** – 모바일폰 교체·수리 전 중요 정보를 삭제하세요.
6. **SMS·OTP 활용** – 휴대폰 문자서비스(SMS)와 일회용비밀번호(OTP)를 이용하세요.
7. **사용환경 임의 변경 금지** – 모바일폰 사용환경을 임의로 변경하지 마세요.
8. **보안업데이트 및 백신 검사** – 보안업데이트를 정기적으로 수행하고 바이러스 검사를 하세요.
9. **잠금기능 설정** – '잠금기능'을 설정하고 '잠금비밀번호'는 수시로 변경하세요.
10. **와이파이 사용 주의** – 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하세요.

## 추가 유의사항

- **비밀번호 설정**: 동일한 숫자가 반복되거나(예: 0000, 1111), 핸드폰 번호나 생일처럼 타인이 쉽게 알아낼 수 있는 번호는 피하는 것이 좋습니다.
- **공공장소 주의**: 공공장소에서 비밀번호가 없는 와이파이를 사용하면 해킹 위험에 노출될 수 있으니 특별히 주의하시기 바랍

## Agentic RAG

- LLM이 RAG의 검색 과정을 스스로 판단하고 제어하는 구조


In [23]:
from typing import TypedDict
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from const.const import create_model, create_embedding

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = create_model()

embeddings = create_embedding()

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5]
)


class State(TypedDict):
    question: str
    search_query: str
    documents: list[Document]
    retry_count: int
    answer: str


def retrieve(state: State):
    print(f"\n[Retrieve] 검색어 '{state['search_query']}'(으)로 문서 검색 중...")
    docs = hybrid_retriever.invoke(state["search_query"])
    print(f"[Retrieve] {len(docs)}개의 연관 문서가 검색되었습니다.")
    return {"documents": docs}


def grade_documents(state: State):
    print("[Grade Documents] LLM을 통해 검색 문서의 연관성을 평가 중...")

    if not state["documents"]:
        if state["retry_count"] >= 2:
            print(
                " -> [재검색 초과] 더 이상의 재검색 없이 답변 생성 단계로 이동합니다."
            )
            return "generate"
        print(" -> [문서 없음] 쿼리 재작성(Rewrite) 노드로 이동합니다.")
        return "rewrite"

    context = "\n".join(doc.page_content for doc in state["documents"])
    prompt = (
        "당신은 검색된 문서가 사용자의 질문과 관련이 있는지 평가하는 검증관입니다.\n"
        f"검색된 문서 내용:\n{context}\n\n"
        "검색된 문서가 질문에 답변하기에 유용한 정보를 포함하고 있다면 'yes', 전혀 관련이 없다면 'no'만 출력하세요."
    )

    response = llm.invoke(
        [SystemMessage(content=prompt), HumanMessage(content=state["question"])]
    )
    grade = str(response.content).strip().lower()

    if "yes" in grade or state["retry_count"] >= 2:
        print(" -> [문서 적합성 확인] 답변 생성(Generate) 노드로 이동합니다.")
        return "generate"
    else:
        print(" -> [문서 연관성 부족] 쿼리 재작성(Rewrite) 노드로 이동합니다.")
        return "rewrite"


def rewrite_query(state: State):
    current_retry = state["retry_count"] + 1
    print(f"[Rewrite] LLM을 이용하여 검색어를 재작성 중... (시도 {current_retry}회차)")

    prompt = (
        "사용자의 질문을 문서 검색기에서 더 잘 검색될 수 있는 핵심 키워드 위주의 검색어로 재작성해주세요.\n"
        "다른 부연설명 없이 오직 재작성된 검색어 단어/문장만 출력하세요."
    )

    response = llm.invoke(
        [SystemMessage(content=prompt), HumanMessage(content=state["question"])]
    )
    new_query = str(response.content).strip()

    print(f"[Rewrite] 최적화된 새 검색어: '{new_query}'")
    return {
        "search_query": new_query,
        "retry_count": current_retry,
    }


def generate(state: State):
    print("[Generate] 검색된 문서를 바탕으로 최종 답변을 생성 중...")
    if not state["documents"]:
        return {"answer": "관련 문서를 찾지 못하여 답변을 생성할 수 없습니다."}

    context = "\n".join(doc.page_content for doc in state["documents"])

    prompt = (
        "아래 제공된 [참고 문서]만을 바탕으로 사용자의 질문에 정확하고 친절하게 답변해주세요.\n\n"
        f"[참고 문서]\n{context}\n\n"
    )

    response = llm.invoke(
        [SystemMessage(content=prompt), HumanMessage(content=state["question"])]
    )
    return {"answer": response.content}


builder = StateGraph(State)

builder.add_node("retrieve", retrieve)
builder.add_node("rewrite", rewrite_query)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")

builder.add_conditional_edges(
    "retrieve",
    grade_documents,
    {
        "rewrite": "rewrite",
        "generate": "generate",
    },
)


builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", END)

graph = builder.compile()

question = "돈 버는 법"

result = graph.invoke(
    {
        "question": question,
        "search_query": question,
        "documents": [],
        "retry_count": 0,
        "answer": "",
    }
)

print("\n=== 최종 Agentic RAG 답변 ===")
print(result["answer"])


[Retrieve] 검색어 '돈 버는 법'(으)로 문서 검색 중...
[Retrieve] 9개의 연관 문서가 검색되었습니다.
[Grade Documents] LLM을 통해 검색 문서의 연관성을 평가 중...
 -> [문서 적합성 확인] 답변 생성(Generate) 노드로 이동합니다.
[Generate] 검색된 문서를 바탕으로 최종 답변을 생성 중...

=== 최종 Agentic RAG 답변 ===
# 돈 버는 법 (참고 문서 기준)

제공된 참고 문서는 특정한 돈 버는 방법보다는 **안전한 투자 원칙과 주의사항**에 대한 내용을 담고 있습니다. 문서에 근거한 내용을 정리해 드리면 다음과 같습니다.

## 1. 워렌 버핏의 5대 투자원칙

'오마하의 현인'이라 불리는 투자의 귀재 워렌 버핏의 투자원칙입니다.

1. **3년 동안 증시가 문을 닫아도 행복할 것이란 자세로 투자에 임하라**
2. **시장은 투자가 잘 됐는지 여부를 알려주지 않는다. 오로지 결과만 말해준다**
3. **안전을 위한 최소한의 여지를 남겨 둬라. 잘못된 길이다 싶으면 빠져 나와라**
4. **똑똑한 사람이 파산하는 가장 일반적인 길은 돈을 빌리는 것이다** (돈을 빌려 투자하지 말라)
5. **감정적으로 주식 거래를 하지 마라. 주식은 아무런 감정이 없다**

## 2. 투자 전 반드시 확인할 사항

- **투자가능기간, 여유자금 규모, 감내할 수 있는 손실규모를 미리 생각**해두어야 합니다.
- 단기간 내 사용처가 정해진 자금(예: 일주일 후 지급해야 하는 부동산매입자금)을 장기 투자하면 조기 해약 위약수수료 등으로 **오히려 손실**이 발생할 수 있습니다.
- 가용투자금액을 확정하지 않아 미수거래·신용거래를 하면 이자 부담과 더 큰 손실이 초래될 수 있으므로 **사전에 가용투자금액을 확정**하는 것이 좋습니다.
- 해외펀드는 환매요청 후 입금까지 보통 7~8일 이상 소요되므로 투자기간 설정 시 참고하세요.
- 원금 손실이 발생하면 경제적으로 애로를 겪을 수 있는 상황이라면 **원금